# Find LinkedIn URLs

### Inputs: 
- `"../derived/companies.txt"`

### Outputs:
- `"../derived/linkedin_company_urls.csv"`

  

### Purpose:

Find LinkedIn URLs of companies mentioned in records through automated Bing search.

To retrieve the company information from Bright Data, we need to know the Company IDs to filter the 56.5M records down to the ones we need.

The Company IDs are the slug used by their respective LinkedIn website. We have a list of (non-standardized) company names from the student dataset. Based on that, we will use the following algorithm to find the slugs:

1. Bing search the company name as it appears in our list with the constraint `site:linkedin.com/company`
2. Validate the URLs (see [02-validate_linkedin_urls.ipynb](02-validate_linkedin_urls.ipynb))
3. Extract validated URLs (see [03-extract_linkedin_ids.ipynb](03-extract_linkedin_ids.ipynb))

We will automate the Bing search using `selenium`.

We are going with Bing because Google's anti-scraping measures make things needlessly complicated.

## Get all URLs

The first step is to get the presumed LinkedIn URL of every company.

In [ ]:
import time
import csv
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from random import uniform
import urllib.parse
import base64


def get_final_url(driver, bing_url):
    """
    Navigate to the Bing tracking URL and get the final destination URL
    """
    try:
        # Store current window handle
        original_window = driver.current_window_handle

        # Open new tab
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[-1])

        # Navigate to the Bing URL
        driver.get(bing_url)

        # Wait a moment for any redirects to complete
        time.sleep(2)

        # Get the final URL
        final_url = driver.current_url

        # Close the tab and switch back to original window
        driver.close()
        driver.switch_to.window(original_window)

        return final_url

    except Exception as e:
        print(f"Error getting final URL: {e}")
        # Make sure we're back on the original window
        try:
            driver.switch_to.window(original_window)
        except:
            pass
        return None


def get_linkedin_company_url(driver, company_name):
    """
    Search Bing for a company's LinkedIn page and return the first result URL.

    Args:
        driver: Selenium WebDriver instance
        company_name (str): The name of the company to search for

    Returns:
        str: The URL of the company's LinkedIn page, or None if not found
    """
    try:
        # Navigate to Bing
        driver.get("https://www.bing.com")

        # Wait for the search box to be available and make sure it's visible
        search_box = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.ID, "sb_form_q"))
        )

        # Construct the search query to limit results to LinkedIn company pages
        search_query = f"{company_name} site:linkedin.com/company"

        # Enter the search query and submit
        search_box.clear()
        # Type the query character by character to mimic human typing
        for char in search_query:
            search_box.send_keys(char)
            time.sleep(uniform(0.05, 0.15))  # Small random delay between keystrokes

        # Add a small pause before hitting Enter
        time.sleep(uniform(0.5, 1.0))
        search_box.send_keys(Keys.RETURN)

        # Wait for search results to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "b_results"))
        )

        # Add a small random delay to avoid being detected as a bot
        time.sleep(uniform(1.0, 3.0))

        # Find the first search result link
        try:
            # Look for the first link in the results that contains linkedin.com/company
            results = driver.find_elements(By.CSS_SELECTOR, "#b_results li.b_algo h2 a")

            for result in results:
                href = get_final_url(
                    driver=driver, bing_url=result.get_attribute("href")
                )
                if "linkedin.com/company" in href:
                    return href

            # If we didn't find a matching link
            print(f"No LinkedIn company page found for {company_name}")
            return None

        except NoSuchElementException:
            print(f"No search results found for {company_name}")
            return None

    except Exception as e:
        print(f"An error occurred while searching for {company_name}: {str(e)}")
        return None


def process_company_list(input_file_path, output_file_path):
    """
    Process a list of companies from a file and save LinkedIn URLs to a CSV.

    Args:
        input_file_path (str): Path to the input file with company names
        output_file_path (str): Path to save the output CSV
    """
    # Set up Chrome options
    chrome_options = Options()
    # Uncomment the line below to run Chrome in headless mode
    # chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--start-maximized")  # Start with maximized window

    # Set up the Chrome driver
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=chrome_options
    )

    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

    # Check if output file exists to determine if we need to continue from where we left off
    companies_processed = set()
    if os.path.exists(output_file_path):
        with open(output_file_path, "r", newline="", encoding="utf-8") as csvfile:
            reader = csv.reader(csvfile)
            next(reader, None)  # Skip header
            for row in reader:
                if row and len(row) > 0:
                    companies_processed.add(row[0])

        print(
            f"Found existing output file with {len(companies_processed)} companies already processed."
        )

    try:
        # Read company names from input file
        with open(input_file_path, "r", encoding="utf-8") as file:
            company_names = [line.strip() for line in file if line.strip()]

        total_companies = len(company_names)
        print(f"Found {total_companies} companies to process.")

        # Open CSV file for writing results
        with open(output_file_path, "a", newline="", encoding="utf-8") as csvfile:
            writer = csv.writer(csvfile)

            # Write header if the file is new
            if not companies_processed:
                writer.writerow(["Company Name", "LinkedIn URL"])

            # Process each company
            for i, company_name in enumerate(company_names):
                if company_name in companies_processed:
                    print(f"Skipping already processed company: {company_name}")
                    continue

                print(f"Processing {i+1}/{total_companies}: {company_name}")

                # Get LinkedIn URL
                linkedin_url = get_linkedin_company_url(driver, company_name)

                # Write result to CSV
                writer.writerow(
                    [company_name, linkedin_url if linkedin_url else "Not found"]
                )
                csvfile.flush()  # Ensure data is written immediately

                # Add to processed set
                companies_processed.add(company_name)

                # Add a delay between requests to avoid rate limiting
                if i < total_companies - 1:  # Don't delay after the last company
                    time.sleep(uniform(2.0, 5.0))

        print(f"Processing complete. Results saved to {output_file_path}")

    except Exception as e:
        print(f"An error occurred during processing: {str(e)}")

    finally:
        # Close the browser
        driver.quit()

In [ ]:
input_file = "../data/derived/companies.txt"
output_file = "../data/derived/linkedin_company_urls.csv"

process_company_list(input_file, output_file)